# AIGC 5005 · Module 05 · Worked example
## A churn model from start to finish

Companion notebook for the matching part of the Module 05 learning content. Read each explanation, **predict** what the next cell will show, run it, and compare. Change values and run again; you cannot break anything.

**Kernel:** choose the `module05-sklearn` conda environment (top right in VS Code). To make it: `conda create -n module05-sklearn python=3.11 ipykernel`, then `conda activate module05-sklearn` and `pip install -r requirements.txt`. This folder also has `churn_model.py` (the training script) and `test_churn_model.py` (its tests).

The script `churn_model.py` is the finished shape of a model project: a function that builds the pipeline, one that evaluates it, and a `main` that splits, cross-validates on the training data, scores the test set **once**, and saves the pipeline with a record of how it was made. This notebook runs it, inspects what it made, and uses the saved model.

## 1. Run the script
The same output every time, because the data, split and folds are seeded.

In [1]:
import churn_model

churn_model.main()

Cross-validated F1 on the training data: 0.370 +/- 0.014
Test set, used once: {'precision': 0.281, 'recall': 0.761, 'f1': 0.411, 'roc_auc': 0.781}
Saved churn_model.joblib and metrics.json


## 2. Read the record it wrote

In [2]:
import json
from pathlib import Path

print(Path("metrics.json").read_text(encoding="utf-8"))

{
  "scikit_learn": "1.8.0",
  "seed": 42,
  "cv_f1_mean": 0.37,
  "test": {
    "precision": 0.281,
    "recall": 0.761,
    "f1": 0.411,
    "roc_auc": 0.781
  }
}


## 3. Use the saved model like an application would
Load it and give it a new customer as a plain table. No scaling, encoding or cleaning is needed because the pipeline carries all of it.

In [3]:
import joblib
import pandas as pd

model = joblib.load("churn_model.joblib")
new = pd.DataFrame([{
    "tenure_months": 3, "monthly_charge": 95.0, "support_calls": 4,
    "contract": "month-to-month", "payment": "cheque",
}])
print(model.predict(new), model.predict_proba(new).round(3))

[1] [[0.071 0.929]]


## 4. Run the tests
The same checks you would run from a terminal with `python -m pytest -v`.

In [4]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-v", "--color=no"], capture_output=True, text=True)
print((result.stdout + result.stderr)[-1500:])

============================= test session starts ==============================
platform linux -- Python 3.11.15, pytest-9.1.1, pluggy-1.7.0 -- /tmp/m5v/bin/python
cachedir: .pytest_cache
rootdir: /home/claude/m5code/run/worked_example
plugins: platformdirs-4.13.5
collecting ... collected 4 items

test_churn_model.py::test_make_customers_is_repeatable PASSED            [ 25%]
test_churn_model.py::test_pipeline_gives_probabilities_between_0_and_1 PASSED [ 50%]
test_churn_model.py::test_pipeline_handles_missing_value_and_unseen_category PASSED [ 75%]
test_churn_model.py::test_evaluate_reports_four_metrics PASSED           [100%]

============================== 4 passed in 1.15s ===============================



## Try it
1. Change `SEED` in `churn_model.py` and rerun. Which numbers change, and by roughly how much?
2. Swap `LogisticRegression` for `RandomForestClassifier(class_weight="balanced", random_state=42)` in `build_pipeline` and compare the test scores.

Generated files (`churn_model.joblib`, `metrics.json`) are ignored by Git.